In [1]:
import sys 

sys.path.append("..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import torch.backends.cudnn as cudnn
import random
from mamba_ssm import Mamba

from thop import clever_format
from ptflops import get_model_complexity_info

import os
os.chdir("/workspace/dehazing")

In [2]:
def set_seed(seed):
    """Sets the seed for reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        cudnn.deterministic = True
        cudnn.benchmark = False

In [3]:
def get_pad_layer(pad_type):
    if(pad_type in ['refl','reflect']):
        PadLayer = nn.ReflectionPad2d
    elif(pad_type in ['repl','replicate']):
        PadLayer = nn.ReplicationPad2da
    elif(pad_type=='zero'):
        PadLayer = nn.ZeroPad2d
    else:
        print(f'Pad type [{pad_type}] not recognized')
    return PadLayer


class AntiAlias_Downsample(nn.Module):
    def __init__(self, channels, pad_type = 'reflect', filt_size = 3, 
                        stride = 2, pad_off = 0):
        super(AntiAlias_Downsample, self).__init__()
        self.filt_size = filt_size
        self.pad_off = pad_off
        self.pad_type = pad_type

        # Asymmetric padding (round up at the top and round down at the bottom)
        # Perfect when kernel size is 2 
        self.pad_sizes = [int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2)),
                          int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2))]
        self.pad_sizes = [pad_size + pad_off for pad_size in self.pad_sizes]
        self.stride = stride 
        self.off = int((self.stride - 1) / 2.)
        self.channels = channels 

        # Define the binomial filter weights
        if(self.filt_size==1):
            a = np.array([1.,])
        elif(self.filt_size==2):
            a = np.array([1., 1.])
        elif(self.filt_size==3):
            a = np.array([1., 2., 1.])
        elif(self.filt_size==4):    
            a = np.array([1., 3., 3., 1.])
        elif(self.filt_size==5):    
            a = np.array([1., 4., 6., 4., 1.])
        elif(self.filt_size==6):    
            a = np.array([1., 5., 10., 10., 5., 1.])
        elif(self.filt_size==7):    
            a = np.array([1., 6., 15., 20., 15., 6., 1.])
            
        # Create a 2D filter by taking the outer product of the 1D filter
        filt = torch.tensor(a[:, None] * a[None, :], dtype = torch.float32)
        filt = filt / torch.sum(filt) # Normalize

        # Reshape to (out_channels, in_channels/groups, kH, kW) for 
        # depthwise convolution
        filt = filt.view(1, 1, filt_size, filt_size)
        filt = filt.repeat(channels, 1, 1, 1)

        # Register as a buffer so PyTorch knows these are NOT trainable parameters
        self.register_buffer('filt', filt)
        self.pad = get_pad_layer(pad_type)(self.pad_sizes)

    def forward(self, inp):
        if (self.filt_size == 1):
            if (self.pad_off == 0):
                return inp[:, :, ::self.stride, ::self.stride] 
            else:
                return self.pad(inp)[:, :, ::self.stride, ::self.stride] 

        else:
            return F.conv2d(self.pad(inp), self.filt, stride = self.stride, groups = inp.shape[1])


class VariantB_AntiAliasedDownsample(nn.Module):
    """
    Anti-Aliased Downsampling (BlurPool) based on Richard Zhang's paper.
    Preserves shift-invariance and prevents high-frequency aliasing.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # 1. Feature Mixing (Stride 1 preserves all spatial information)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)
        # 2. Anti-aliased spatial reduction (Low-pass filter + subsampling)
        # NOTE: Make sure your AntiAlias_Downsample class is defined in the script!
        self.aa_down = AntiAlias_Downsample(channels=dim_out, filt_size=3, stride=2)

    def forward(self, x):
        return self.aa_down(self.conv(x))

In [4]:
class LocalFeatureExtractor(nn.Module):
    """
    Refined for Mamba Block Integration.
    Focuses on edge-preservation and local consistency.
    """
    def __init__(self, dim, kernel_size=3, dilation=1):
        super().__init__()
        padding = (dilation * (kernel_size - 1)) // 2
        
        # We use a Depthwise-Pointwise structure to keep it fast
        self.conv = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=kernel_size, padding=padding, 
                      dilation=dilation, groups=dim), # Local spatial context
            nn.BatchNorm2d(dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, kernel_size=1), # Inter-channel communication
            nn.BatchNorm2d(dim)
        )

    def forward(self, x):
        # We add the input back (Residual) so that even if the gate is 
        # closed, the original features aren't lost.
        return x + self.conv(x)

In [5]:
class PhysBiMambaBlock(nn.Module):
    """
    Bidirectional Mamba Block (BiMamba)
    Scans the image Forward AND Backward so the top-left pixel
    can 'see' the bottom-right pixel.|
    """
    def __init__(self, dim, dropout = 0.05):
        super().__init__()
        self.norm = nn.LayerNorm(dim)

        # Note: In true VMamba, they share the input projection layer to save memory. 
        # But keeping 4 separate Mambas is fine if you have the GPU RAM.
        self.mamba_h_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_h_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        
        # Fuses Fwd+Bwd direction
        self.fusion_proj = nn.Linear(dim, dim)

        # Smoothing to explicitly destroy 1D streaking artifacts
        self.spatial_smoothing = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)
        
        self.local_conv = LocalFeatureExtractor(dim, kernel_size=3, dilation=1)
        
        # Optional: A Gate to let the network choose emphasis
        self.mixer = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )
        
        # Initialize mixer bias negatively so local_conv is favored early in training
        nn.init.constant_(self.mixer[0].bias, -1.0)
        
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, t_emb=None):
        B, C, H, W = x.shape
        residual = x
        
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm = self.norm(x_flat)

        if t_emb is not None:
            scale, shift = t_emb.chunk(2, dim=1)
            x_norm = x_norm * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

        # ---------------------------------------------------------
        # 2. HORIZONTAL SCANS (Raster Order)
        # ---------------------------------------------------------
        # Forward ->
        out_h_fwd = self.mamba_h_fwd(x_norm)
        
        # Backward <-
        x_flip = torch.flip(x_norm, dims=[1])
        out_h_bwd = self.mamba_h_bwd(x_flip)
        out_h_bwd = torch.flip(out_h_bwd, dims=[1]) # Flip back

        # ---------------------------------------------------------
        # 3. VERTICAL SCANS (Column-Major Order)
        # ---------------------------------------------------------
        # Reshape to Image -> Transpose (Swap H and W) -> Flatten
        # Result: (B, W*H, C). Now 'neighbors' in seq are vertical neighbors.
        x_v_img = x_norm.view(B, H, W, C).permute(0, 2, 1, 3) 
        x_v_flat = x_v_img.flatten(1, 2)
        
        # Down v
        out_v_fwd = self.mamba_v_fwd(x_v_flat)
        
        # Up ^
        x_v_flip = torch.flip(x_v_flat, dims=[1])
        out_v_bwd = self.mamba_v_bwd(x_v_flip)
        out_v_bwd = torch.flip(out_v_bwd, dims=[1])
        
        # Un-Transpose Vertical Outputs back to Horizontal Order
        # (B, W*H, C) -> (B, W, H, C) -> (B, H, W, C) -> (B, L, C)
        out_v_fwd = out_v_fwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        out_v_bwd = out_v_bwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        
        ## ---------------------------------------------------------
        # 4. Global Fusion
        # ---------------------------------------------------------
        # Combine all 4 views of the image
        # WE are treating the outputs of the 4 directions seperate.
        # It mixes the channels of the 4 scans for a single pixel, but it does not communicate with neighboring pixels. 
        # The model simply overlays the horizontal streaks and vertical streaks on top of each other.
        # global_feat = self.fusion_linear(
        #     torch.cat([out_h_fwd, out_h_bwd, out_v_fwd, out_v_bwd], dim=-1)
        # )

        global_feat = out_h_fwd + out_h_bwd + out_v_fwd + out_v_bwd
        global_feat = self.fusion_proj(global_feat)

        # ---------------------------------------------------------
        # 5. NEW: Spatial Smoothing to remove streaks
        # ---------------------------------------------------------
        global_feat_img = global_feat.transpose(1, 2).view(B, C, H, W)
        global_feat_img = self.spatial_smoothing(global_feat_img)
        global_feat = global_feat_img.flatten(2).transpose(1, 2)

        # ---------------------------------------------------------
        # 6. Local Branch (Conv)
        # ---------------------------------------------------------
        # Reshape for Conv2d
        x_img_norm = x_norm.transpose(1, 2).view(B, C, H, W)
        local_feat = self.local_conv(x_img_norm)
        local_feat = local_feat.flatten(2).transpose(1, 2)

        
        # ---------------------------------------------------------
        # 6. Gated Output
        # ---------------------------------------------------------
        combined = torch.cat([global_feat, local_feat], dim=-1)
        z = self.mixer(combined)
        
        fused = global_feat * z + local_feat * (1 - z)
        
        x_out = self.out_proj(fused)
        
        # Reshape to (B, C, H, W) for residual add
        x_out = x_out.transpose(1, 2).view(B, C, H, W)
        x_out = self.dropout(x_out)
        
        return residual + x_out


In [6]:
class BilinearUpsample(nn.Module):
    """
    Used by BOTH variants. Guarantees no checkerboard artifacts during decoding.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        return self.conv(self.up(x))

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AblationPhysicsEstimator(nn.Module):
    """
    Refactored for Structural Sharpness:
    - Replaces Dilation with Depthwise Separable Bottleneck (Preserves local edges).
    - Swaps Bilinear Upsampling for PixelShuffle (Prevents interpolation blur).
    - Uses a Gated Residual connection for the Transmission map.
    """
    def __init__(self, in_channels=4, base_dim=32):
        super().__init__()
        self.variant = 'S_Sharp' # 'S' for Structural Sharpness
        
        # 1. Initial Encoder (RGB + DCP)
        self.init_conv = nn.Conv2d(in_channels, base_dim, kernel_size=3, padding=1)
        self.enc1 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)

        # 2. Downsampling (Anti-Aliased)
        self.down1 = VariantB_AntiAliasedDownsample(base_dim, base_dim * 2)
        self.down2 = VariantB_AntiAliasedDownsample(base_dim * 2, base_dim * 4)

        # 3. Dilated Bottleneck (1 -> 2 -> 1 Dilation Pattern)
        # This expands the receptive field to capture "haze context" 
        # while maintaining local edge precision.
        self.bottleneck = nn.Sequential(
            # Dilation 1: Local context
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Dilation 2: Mid-range context (Captures haze gradients)
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Dilation 1: Refine back to local structural details
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )        
        
        # 4. Sharp Decoder (Using PixelShuffle to avoid interpolation blur)
        # Note: PixelShuffle(upscale_factor=2) reduces channels by 4x
        self.up1_ps = nn.Sequential(
            nn.Conv2d(base_dim * 4, base_dim * 8, kernel_size=1),
            nn.PixelShuffle(2)
        ) # Result: base_dim * 2 channels
        
        self.dec1_fusion = nn.Conv2d(base_dim * 4, base_dim * 2, kernel_size=1) 
        self.dec1 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)
        
        self.up2_ps = nn.Sequential(
            nn.Conv2d(base_dim * 2, base_dim * 4, kernel_size=1),
            nn.PixelShuffle(2)
        ) # Result: base_dim channels
        
        self.dec2_fusion = nn.Conv2d(base_dim * 2, base_dim, kernel_size=1)
        self.dec2 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        
        # 5. Output Heads
        self.t_head_residual = nn.Sequential(
            nn.Conv2d(base_dim, 1, kernel_size=3, padding=1), 
            nn.Tanh() 
        )

        self.A_head_spatial = nn.Sequential(
            nn.Conv2d(base_dim, 3, kernel_size=3, padding=1), 
            nn.Sigmoid() 
        )

        # self.A_head_global = nn.Sequential(
        #     nn.AdaptiveAvgPool2d(1),
        #     nn.Flatten(),
        #     nn.Linear(base_dim, base_dim // 2),
        #     nn.ReLU(inplace=True),
        #     nn.Linear(base_dim // 2, 3),
        #     nn.Sigmoid()
        # )

    def forward(self, hazy_img, t_dcp):
        x = torch.cat([hazy_img, t_dcp], dim=1)

        # Encoder
        x = F.relu(self.init_conv(x))
        e1 = F.relu(self.enc1(x))
        d1 = self.down1(e1)
        e2 = F.relu(self.enc2(d1))
        d2 = self.down2(e2)

        # Bottleneck (Preserves structural edges)
        b = self.bottleneck(d2)
        
        # Decoder 1: PixelShuffle + Skip Connection
        u1 = torch.cat([self.up1_ps(b), e2], dim=1)
        u1 = F.relu(self.dec1_fusion(u1))
        u1 = F.relu(self.dec1(u1))
        
        # Decoder 2: PixelShuffle + Skip Connection
        u2 = torch.cat([self.up2_ps(u1), e1], dim=1)
        u2 = F.relu(self.dec2_fusion(u2))
        u2 = F.relu(self.dec2(u2))

        # --- FINAL OUTPUTS ---
        t_residual = self.t_head_residual(u2)
        
        # Final Transmission = DCP Prior + Learned Residual
        t_final = torch.clamp(t_dcp + t_residual, 0.0, 1.0)
        A_spatial = self.A_head_spatial(u2)

        return t_final, A_spatial

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stage1 = AblationPhysicsEstimator().to(device)
x = torch.randn(1, 3, 256, 256).to(device)
t_dcp = torch.randn(1, 1, 256, 256).to(device)
t_final, A_spatial = stage1(x, t_dcp)

print(f"t map shape: {t_final.shape}")
print(f"A spatial shape: {A_spatial.shape}")


t map shape: torch.Size([1, 1, 256, 256])
A spatial shape: torch.Size([1, 3, 256, 256])


### Loss Functions

In [13]:
import torch.fft 
import os
import torch
import torch.nn.functional as F
import torchvision
import torch.fft
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure


def get_sobel_edges(img):
    """ Helper to extract edges using Sobel filters """
    kernel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], device=img.device).float().unsqueeze(0).unsqueeze(0)
    kernel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], device=img.device).float().unsqueeze(0).unsqueeze(0)
    
    # Repeat for channels if necessary, but transmission is 1-ch
    edges_x = F.conv2d(img, kernel_x, padding=1)
    edges_y = F.conv2d(img, kernel_y, padding=1)
    return torch.sqrt(edges_x**2 + edges_y**2 + 1e-6)


def get_fft_spectrum(img):
    """ Helper to visualize the frequency spectrum """
    # Compute 2D FFT and shift low frequencies to center
    fft = torch.fft.fft2(img)
    fft_shift = torch.fft.fftshift(fft)

    # Magnitude in log scale for visualization
    magnitude = torch.log(torch.abs(fft_shift) + 1e-6)

    # Normalize to [0, 1] for TensorBoard
    mag_min, mag_max = magnitude.min(), magnitude.max()

    return (magnitude - mag_min) / (mag_max - mag_min + 1e-6)


In [14]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super(CharbonnierLoss, self). __init__()
        self.eps = eps

    def forward(self, x, y):
        diff = x - y
        loss = torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))
        return loss


In [15]:
class Stage1_RESIDELoss_Unified(nn.Module):
    def __init__(self, w_t=1.0, w_recon=0.5, w_struct=2.5, w_grad=1.0):
        super().__init__()
        self.charbonnier = CharbonnierLoss()
        self.w_t = w_t
        self.w_recon = w_recon
        self.w_struct = w_struct
        self.w_grad = w_grad # New weight for gradient direction
        
        # Pre-define kernels
        self.register_buffer('sobel_x', 
                             torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).float().view(1,1,3,3))
        self.register_buffer('sobel_y', 
                             torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).float().view(1,1,3,3))
        self.register_buffer('laplacian', 
                             torch.tensor([[-1,-1,-1], [-1,8,-1], [-1,-1,-1]]).float().view(1,1,3,3))

    def get_gradients(self, x):
        grad_x = F.conv2d(x, self.sobel_x, padding=1)
        grad_y = F.conv2d(x, self.sobel_y, padding=1)
        # Magnitude
        mag = torch.sqrt(grad_x**2 + grad_y**2 + 1e-6)
        return grad_x, grad_y, mag

    def forward(self, pred_t_map, pred_A, gt_t_map, clean_img_01, hazy_img_01):
        # 1. Pixel-wise Anchor (L1-style via Charbonnier)
        loss_t = self.charbonnier(pred_t_map, gt_t_map)

        # 2. Advanced Structural & Gradient Loss
        p_gx, p_gy, p_mag = self.get_gradients(pred_t_map)
        g_gx, g_gy, g_mag = self.get_gradients(gt_t_map)
        
        # A. Laplacian for high-frequency "crispness"
        p_lap = F.conv2d(pred_t_map, self.laplacian, padding=1)
        g_lap = F.conv2d(gt_t_map, self.laplacian, padding=1)
        
        # B. Gradient Magnitude Loss (Standard Structural)
        loss_struct = self.charbonnier(p_mag, g_mag) + 0.5 * self.charbonnier(p_lap, g_lap)
        
        # C. Gradient Direction Loss (Crucial for Sharpness)
        # Punishes the model if the edge "points" in the wrong direction
        loss_grad_dir = torch.mean(torch.abs(p_gx - g_gx) + torch.abs(p_gy - g_gy))

        # 3. Physics Constraint
        if pred_A.dim() == 2:
            A_final = pred_A.view(pred_A.size(0), 3, 1, 1)
        else:
            A_final = pred_A

        # I = J * t + A * (1 - t)
        I_recon = (clean_img_01 * pred_t_map) + (A_final * (1.0 - pred_t_map))
        loss_recon = self.charbonnier(I_recon, hazy_img_01)

        # 4. Total Loss Summation
        total_loss = (self.w_t * loss_t) + \
                     (self.w_struct * loss_struct) + \
                     (self.w_grad * loss_grad_dir) + \
                     (self.w_recon * loss_recon)

        return total_loss, {
            "Total": total_loss.item(),
            "Pixel": loss_t.item(), 
            "Structural": loss_struct.item(),
            "Grad_Dir": loss_grad_dir.item(),
            "Recon": loss_recon.item()
        }

In [31]:
import torch
import torch.nn.functional as F
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio

def gmsd_loss(pred, gt, c=0.0026):
    """
    Optimized Gradient Magnitude Similarity Deviation (GMSD).
    Expects tensors of shape [B, C, H, W] normalized to [0, 1].
    """
    B, C, H, W = pred.shape
    
    # 1. Define Prewitt Filters (Standard for GMSD)
    # Using 1/3 as per original paper; repeated for each input channel
    kernel_x = torch.tensor([[[[1/3, 0, -1/3], [1/3, 0, -1/3], [1/3, 0, -1/3]]]], 
                            device=pred.device, dtype=pred.dtype)
    kernel_x = kernel_x.repeat(C, 1, 1, 1) # Support RGB or Grayscale
    kernel_y = kernel_x.transpose(2, 3)

    # 2. Compute Gradients via Depthwise Convolution (Faster)
    # groups=C ensures each channel is processed independently
    p_gx = F.conv2d(pred, kernel_x, padding=1, groups=C)
    p_gy = F.conv2d(pred, kernel_y, padding=1, groups=C)
    p_mag = torch.sqrt(p_gx**2 + p_gy**2 + 1e-12)

    g_gx = F.conv2d(gt, kernel_x, padding=1, groups=C)
    g_gy = F.conv2d(gt, kernel_y, padding=1, groups=C)
    g_mag = torch.sqrt(g_gx**2 + g_gy**2 + 1e-12)

    # 3. Calculate Gradient Magnitude Similarity (GMS)
    # GMS is 1.0 where gradients are identical, lower elsewhere
    num = 2 * p_mag * g_mag + c
    den = p_mag**2 + g_mag**2 + c
    gms = num / den

    # 4. Standard Deviation (The "D" in GMSD)
    # The original paper suggests pooling/averaging the GMS map first 
    # if it's high res, but standard SD on the whole map is common.
    # We calculate SD over (C, H, W) for each image in batch
    score = torch.std(gms, dim=[1, 2, 3]) 
    
    return score.mean()


def evaluate_transmission_quality(pred_t, gt_t):
    """
    pred_t, gt_t: Tensors of shape [B, 1, H, W] in range [0, 1]
    """
    # 1. Standard PSNR (Pixel Accuracy)
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(pred_t.device)
    psnr_val = psnr_metric(pred_t, gt_t)

    # 2. SSIM (Structural Accuracy - fixes the PSNR paradox)
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(pred_t.device)
    ssim_val = ssim_metric(pred_t, gt_t)

    # 3. GMSD (Gradient/Edge Accuracy - Custom PyTorch implementation)
    gmsd_val = gmsd_loss(pred_t, gt_t)

    # 4. Total Variation (Measures "Cleanliness" / lack of noise)
    def total_variation(img):
        diff_h = torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).sum()
        diff_w = torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).sum()
        return diff_h + diff_w

    tv_val = total_variation(pred_t)

    return {
        "PSNR (HIGH is good)": psnr_val.item(),
        "SSIM (HIGH is good)": ssim_val.item(),
        "GMSD (LOW is good)": gmsd_val.item(),
        "TV_Smoothness": tv_val.item()
    }

In [32]:
pred = torch.randn(1, 3, 256, 256)
gt = torch.randn(1, 3, 256, 256)

gmsd_loss(pred, gt)

tensor(0.2230)

In [33]:
def get_fft_spectrum(img):
    """ Helper to visualize the frequency spectrum """
    # Compute 2D FFT and shift low frequencies to center
    fft = torch.fft.fft2(img)
    fft_shift = torch.fft.fftshift(fft)

    # Magnitude in log scale for visualization
    magnitude = torch.log(torch.abs(fft_shift) + 1e-6)

    # Normalize to [0, 1] for TensorBoard
    mag_min, mag_max = magnitude.min(), magnitude.max()

    return (magnitude - mag_min) / (mag_max - mag_min + 1e-6)

## Getting the dataset

In [34]:
from data.utils import get_haze_transforms, partition_dataset
from torch.utils.data import Subset
from losses import CharbonnierLoss
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

In [35]:
class RESIDE_Indoor(Dataset):
    def __init__(self, dataset_path, transform=None):
        self.root_dir = Path(dataset_path)
        self.metadata_csv = pd.read_csv(self.root_dir / "metadata.csv")

        self.transform = transform
        self.data = []
        
        for idx, row in self.metadata_csv.iterrows():
            clean_path = self.root_dir / row["clear_image_path"]
            hazy_paths_str = row["hazy_image_paths"]
            hazy_image_paths = [
                path.strip()
                for path in hazy_paths_str.strip("[]").replace("'", "").split(",")
            ]
            list_hazy_paths = [
                self.root_dir / hazy_path for hazy_path in hazy_image_paths
            ]
            
            for hazy_path in list_hazy_paths:
                # --- NEW LOGIC: Deduce the Transmission Map Path ---
                # Example: hazy_path.name is "1_1_0.90179.png"
                hazy_filename = hazy_path.name
                parts = hazy_filename.split('_')
                
                # Reconstruct trans filename: "1_1.png"
                if len(parts) >= 2:
                    trans_filename = f"{parts[0]}_{parts[1]}.png"
                else:
                    trans_filename = hazy_filename # Fallback just in case
                
                trans_path = self.root_dir / "trans" / trans_filename
                # ---------------------------------------------------

                data_item = {
                    "index": idx, 
                    "clean": clean_path, 
                    "hazy": hazy_path,
                    "trans": trans_path # Store the trans path
                }

                self.data.append(data_item)

    def __repr__(self):
        return "RESIDE Indoor"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]
        clean_path = data_item["clean"]
        hazy_path = data_item["hazy"]
        trans_path = data_item["trans"]

        try:
            clean_img = Image.open(clean_path).convert("RGB")
            hazy_img = Image.open(hazy_path).convert("RGB")
            # Load transmission map as Grayscale ("L")
            trans_img = Image.open(trans_path).convert("L") 
        except FileNotFoundError:
            print(f"Error: Missing image file at {clean_path}, {hazy_path}, or {trans_path}. Skipping")
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            # IMPORTANT WARNING: 
            # If your 'get_haze_transforms' function only expects 2 inputs, 
            # you must update it to accept and return 3 inputs!
            clean_img, hazy_img, trans_img = self.transform(clean_img, hazy_img, trans_img)
        else:
            # Fallback tensorization
            clean_img = (
                torch.as_tensor(np.array(clean_img)).permute(2, 0, 1).float() / 255.0
            )
            hazy_img = (
                torch.as_tensor(np.array(hazy_img)).permute(2, 0, 1).float() / 255.0
            )
            # Add channel dimension to grayscale image (H, W) -> (1, H, W)
            trans_img = (
                torch.as_tensor(np.array(trans_img)).unsqueeze(0).float() / 255.0
            )

        return hazy_img, clean_img, trans_img

In [36]:
def get_reside_indoor_transforms(resize_size: int = 256):
    """
    Returns both train and val transforms strictly tuned for RESIDE-INDOOR.
    Safely handles the optional 3rd 'trans' (transmission) map.
    """
    
    # 1. Base Formatting: Convert to tensor [0, 1] -> Normalize to [-1, 1]
    to_tensor_norm = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])

    # 2. Geometric Sync: Applied equally to clean, hazy, and trans to preserve alignment
    geometric_sync = v2.Compose([
        v2.RandomCrop(resize_size, pad_if_needed=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),
    ])

    # 3. Color Jitter: Tuned specifically for RESIDE-INDOOR lighting
    color_jitter = v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.01)

    # --- TRAIN TRANSFORM ---
    def train_transform(clean, hazy, trans=None):
        # A. Apply spatial changes synchronously
        if trans is not None:
            clean, hazy, trans = geometric_sync(clean, hazy, trans)
        else:
            clean, hazy = geometric_sync(clean, hazy)

        # B. Apply color distortion ONLY to the hazy image
        hazy = color_jitter(hazy)

        # C. Tensor & Normalization
        clean, hazy = to_tensor_norm(clean), to_tensor_norm(hazy)
        
        if trans is not None:
            # Trans stays [0, 1] for physics math, NO normalization!
            trans = v2.functional.to_dtype(v2.functional.to_image(trans), torch.float32, scale=True)
            return clean, hazy, trans
            
        return clean, hazy

    # --- VAL TRANSFORM ---
    def val_transform(clean, hazy, trans=None):
        # Validation just normalizes. NO cropping or flipping.
        clean, hazy = to_tensor_norm(clean), to_tensor_norm(hazy)
        
        if trans is not None:
            trans = v2.functional.to_dtype(v2.functional.to_image(trans), torch.float32, scale=True)
            return clean, hazy, trans
            
        return clean, hazy

    return train_transform, val_transform

In [37]:
set_seed(42)

resolution = 256
verbose = True
num_subset_samples = 500

train_transform, val_transform = get_reside_indoor_transforms(resize_size = resolution)

data_path = "dataset/indoor-training-set/"
dataset = RESIDE_Indoor(dataset_path=data_path)

indices = torch.randperm(len(dataset))[:num_subset_samples].tolist()
subset_dataset = Subset(dataset, indices)

train_dataset, val_dataset = partition_dataset(
    subset_dataset,
    train_transform, 
    val_transform,
    train_ratio = 0.8
)

print(f"Total Subset Size: {len(subset_dataset)}")
print(f"Training Set Size: {len(train_dataset)}")
print(f"Validation Set Size: {len(val_dataset)}")

Total Subset Size: 500
Training Set Size: 400
Validation Set Size: 100


In [38]:
def total_variation(img):
    """ Measures 'Cleanliness' / lack of noise """
    diff_h = torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).sum(dim=[1,2,3])
    diff_w = torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).sum(dim=[1,2,3])
    return (diff_h + diff_w).mean()
    

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BatchedDCP(nn.Module):
    def __init__(self, patch_size=15, omega=0.95, top_p=0.001):
        super().__init__()
        self.patch_size = patch_size
        self.omega = omega
        self.top_p = top_p

    def get_dark_channel(self, I):
        # 1. Min over RGB channels
        min_c, _ = torch.min(I, dim=1, keepdim=True)
        # 2. Min over spatial patch
        pad = self.patch_size // 2
        dc = -F.max_pool2d(-min_c, kernel_size=self.patch_size, stride=1, padding=pad)
        return dc

    def get_atmospheric_light(self, I, dc):
        B, C, H, W = I.shape
        A = torch.zeros(B, C, 1, 1, device=I.device)
        num_pixels = H * W
        num_top = max(int(num_pixels * self.top_p), 1)
        
        for i in range(B):
            dc_flat = dc[i, 0].view(-1)
            I_flat = I[i].view(C, -1)
            _, indices = torch.topk(dc_flat, num_top)
            A[i, :, 0, 0] = torch.mean(I_flat[:, indices], dim=1)
        return A

    def forward(self, I):
        dc = self.get_dark_channel(I)
        A = self.get_atmospheric_light(I, dc)
        
        # Normalize by A and get dark channel of normalized image
        I_norm = I / (A + 1e-6)
        t_map = 1.0 - self.omega * self.get_dark_channel(I_norm)
        
        return torch.clamp(t_map, 0.1, 1.0), torch.clamp(A, 0.0, 1.0)

In [46]:
def train_stage1_dcp_residual(train_loader, val_loader,
                                 num_epochs=30, lr=1e-3, accum_iter=4, 
                                 w_t=1.0, w_recon=0.5, w_struct=1.5):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Setup Naming & Logging
    run_name = f"DCP_Refine_V3_VariantD_wt{w_t}_wr{w_recon}_ws{w_struct}"
    print(f"🚀 Starting Stage 1 Training: {run_name} | Effective BS: {train_loader.batch_size * accum_iter}")
    
    log_dir = os.path.join("runs", "Stage1_Final", run_name)
    writer = SummaryWriter(log_dir=log_dir)

    # 3. Model, Prior, & Optimization
    # Fixed mathematical prior (No gradients)
    dcp_estimator = BatchedDCP(patch_size=15).to(device)
    dcp_estimator.eval() 
    
    # 4-channel input (RGB + DCP_t) U-Net
    model = AblationPhysicsEstimator(in_channels=4, base_dim=32).to(device)
    
    # Adaptive Loss (Handles Spatial or Global A)
    criterion = Stage1_RESIDELoss_Unified(w_t=w_t, w_recon=w_recon, w_struct=w_struct).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    global_step = 0

    for epoch in range(num_epochs):
        # ==========================================
        #               TRAINING PHASE
        # ==========================================
        model.train()
        train_iterator = tqdm(train_loader, desc=f"Train Ep {epoch+1}/{num_epochs}", leave=True)
        optimizer.zero_grad()
        
        for i, (hazy, clean, trans_gt) in enumerate(train_iterator):
            hazy, clean, trans_gt = hazy.to(device), clean.to(device), trans_gt.to(device)

            # A. Extract DCP Prior as a baseline (detached from gradient graph)
            with torch.no_grad():
                t_dcp_prior, _ = dcp_estimator(hazy)

            # B. Forward Pass: Refine the DCP baseline
            pred_t, pred_A_spatial = model(hazy, t_dcp_prior)
            
            # C. Loss Calculation
            loss, loss_dict = criterion(pred_t, pred_A_spatial, trans_gt, clean, hazy)

            # D. Gradient Accumulation
            (loss / accum_iter).backward()

            if (i + 1) % accum_iter == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            if global_step % 20 == 0:
                for k, v in loss_dict.items():
                    writer.add_scalar(f"Train_Loss/{k}", v, global_step)

            global_step += 1
            train_iterator.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()
        writer.add_scalar("Hyperparams/Learning_Rate", scheduler.get_last_lr()[0], epoch)
        
        # ==========================================
        #              VALIDATION PHASE
        # ==========================================
        model.eval()
        val_psnr_acc = 0.0
        val_ssim_acc = 0.0
        val_gmsd_acc = 0.0
        recon_psnr_accum = 0.0
        num_val_batches = len(val_loader)

        val_iterator = tqdm(val_loader, desc="Validating", leave=False)
        with torch.no_grad():
            for v_hazy, v_clean, v_trans in val_iterator:
                v_hazy, v_clean, v_trans = v_hazy.to(device), v_clean.to(device), v_trans.to(device)
                
                # Prediction
                v_t_dcp, _ = dcp_estimator(v_hazy)
                p_t, p_A_spatial = model(v_hazy, v_t_dcp)
                p_t, p_A_spatial = torch.clamp(p_t, 0, 1), torch.clamp(p_A_spatial, 0, 1)

                metrics = evaluate_transmission_quality(p_t, v_trans)

                # Accumulate metrics
                val_psnr_acc += metrics["PSNR (HIGH is good)"]
                val_ssim_acc += metrics["SSIM (HIGH is good)"]
                val_gmsd_acc += metrics["GMSD (LOW is good)"]
                
                # Physics Check: I = Jt + A(1-t)
                A_eff = p_A_spatial.view(-1, 3, 1, 1) if p_A_spatial.dim() == 2 else p_A_spatial
                I_recon = v_clean * p_t + A_eff * (1.0 - p_t)
                I_recon = torch.clamp(I_recon, 0.0, 1.0)
                
                recon_mse = F.mse_loss(I_recon, v_hazy)
                recon_psnr_accum += 10 * torch.log10(1.0 / (recon_mse + 1e-8)).item()

            avg_psnr = val_psnr_acc / num_val_batches
            avg_ssim = val_ssim_acc / num_val_batches
            avg_gmsd = val_gmsd_acc / num_val_batches
            avg_recon_psnr = recon_psnr_accum / len(val_loader)

            # Log to TensorBoard
            writer.add_scalar("Val_Metrics/Avg_PSNR", avg_psnr, epoch)
            writer.add_scalar("Val_Metrics/Avg_SSIM", avg_ssim, epoch)
            writer.add_scalar("Val_Metrics/Avg_GMSD", avg_gmsd, epoch)
            writer.add_scalar("Diagnostics/Recon_PSNR", avg_recon_psnr, epoch)

            # ==========================================
            #       QUALITATIVE DIAGNOSTIC GRID
            # ==========================================
            sample_hazy, sample_clean, sample_trans = next(iter(val_loader))
            s_h, s_c, s_t_gt = sample_hazy[:4].to(device), sample_clean[:4].to(device), sample_trans[:4].to(device)

            with torch.no_grad():
                s_t_dcp, _ = dcp_estimator(s_h)
                s_t_pred, s_A_spatial = model(s_h, s_t_dcp)
                s_t_pred = torch.clamp(s_t_pred, 0, 1)

            # Visual stacking and grid generation
            vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
            vis_pred_t = s_t_pred.repeat(1, 3, 1, 1)
            vis_dcp_t = s_t_dcp.repeat(1, 3, 1, 1)
            abs_err = torch.abs(s_t_pred - s_t_gt)
            err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)

            # Compute FFT Spectrums
            fft_gt = get_fft_spectrum(s_t_gt).repeat(1, 3, 1, 1)
            fft_pred = get_fft_spectrum(s_t_pred).repeat(1, 3, 1, 1)

            # Row 1: Hazy | Row 2: DCP (Prior) | Row 3: Pred (Refined) | Row 4: GT | Row 5: Error
            full_vis_stack = torch.cat([
                s_h, 
                vis_dcp_t, 
                vis_pred_t, 
                vis_gt_t, 
                err_edges,
                fft_gt, 
                fft_pred
            ], dim=0)
            grid = torchvision.utils.make_grid(full_vis_stack, nrow=4, normalize=False)
            writer.add_image('Diagnostic_Grid/Evolution', grid, epoch)

            # Final Recon Check
            A_eff = s_A_spatial.view(-1, 3, 1, 1) if s_A_spatial.dim() == 2 else s_A_spatial       
            s_recon = torch.clamp(s_c * s_t_pred + A_eff * (1.0 - s_t_pred), 0, 1)
            recon_grid = torchvision.utils.make_grid(torch.cat([s_h, s_recon], dim=0), nrow=4, normalize=True)
            writer.add_image('Reconstruction/Comparison', recon_grid, epoch)

        if epoch == num_epochs - 1 or epoch % 10 == 0:
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(model.state_dict(), f"checkpoints/{run_name}_ep{epoch}.pth")

    writer.close()
    print(f"✅ Stage 1 Complete | PSNR: {avg_psnr:.2f} | SSIM: {avg_ssim:.4f} | GMSD: {avg_gmsd:.4f}")
    return model

In [47]:
train_loader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 8, shuffle = False)

model = train_stage1_dcp_residual(train_loader, val_loader,
                                 num_epochs=1, lr=1e-3, accum_iter=4, 
                                 w_t=1.0, w_recon=0.5, w_struct=1.5)

🚀 Starting Stage 1 Training: DCP_Refine_Test_VariantD_wt1.0_wr0.5_ws1.5 | Effective BS: 32


Train Ep 1/1:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Stage 1 Complete | PSNR: 17.41 | SSIM: 0.8437 | GMSD: 0.1644


In [51]:
def benchmark_dcp_residual_model(checkpoint_path, val_loader, log_base="runs/Benchmarks/DCP_Residual"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Setup Naming
    v_name = "DCP_V2_Residual_Refinement"
    writer = SummaryWriter(log_dir=os.path.join(log_base, v_name))

    # 2. Initialize Models
    # Prior module
    dcp_estimator = BatchedDCP(patch_size=15).to(device)
    dcp_estimator.eval()
    
    # Estimator module (AblationPhysicsEstimator expects 4 channels)
    model = AblationPhysicsEstimator(in_channels=4, base_dim=32).to(device)
    
    # Load weights
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    # 3. Get Fixed Batch (Seed 42 for consistency across model versions)
    g = torch.Generator()
    g.manual_seed(42) 
    fixed_loader = DataLoader(val_loader.dataset, batch_size=4, shuffle=True, generator=g)
    s_h, s_c, s_t_gt = next(iter(fixed_loader))
    s_h, s_c, s_t_gt = s_h.to(device), s_c.to(device), s_t_gt.to(device)

    # 4. Inference
    with torch.no_grad():
        # A. Compute Prior
        s_t_dcp, _ = dcp_estimator(s_h)
        
        # B. Model Prediction (Refinement)
        s_t_pred, s_A_spatial = model(s_h, s_t_dcp)
        s_t_pred = torch.clamp(s_t_pred, 0.0, 1.0)
        s_A_spatial = torch.clamp(s_A_spatial, 0.0, 1.0)
        
        # C. Physical Reconstruction (Physics Check)
        # Using predicted t and ground truth clean image to re-synthesize haze
        s_h_recon = s_c * s_t_pred + s_A_spatial * (1.0 - s_t_pred)
        s_h_recon = torch.clamp(s_h_recon, 0.0, 1.0)

        # 5. CALCULATE METRICS
        # A. Reconstruction PSNR
        recon_mse = F.mse_loss(s_h_recon, s_h)
        recon_psnr = 10 * torch.log10(1.0 / (recon_mse + 1e-8))
        
        # B. Transmission PSNR
        trans_mse = F.mse_loss(s_t_pred, s_t_gt)
        trans_psnr = 10 * torch.log10(1.0 / (trans_mse + 1e-8))
        
        # C. Edge PSNR
        gt_edges = get_sobel_edges(s_t_gt)
        edge_mask = (gt_edges > 0.1).float() 
        edge_mse = F.mse_loss(s_t_pred * edge_mask, s_t_gt * edge_mask)
        edge_psnr = 10 * torch.log10(1.0 / (edge_mse + 1e-8))

    # 6. Build Diagnostic Rows
    vis_dcp_t = s_t_dcp.repeat(1, 3, 1, 1)
    vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
    vis_pred_t = s_t_pred.repeat(1, 3, 1, 1)

    # Edge Error Visualization
    abs_err = torch.abs(s_t_pred - s_t_gt)
    err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)
    err_edges = err_edges / (err_edges.max() + 1e-8)

    # FFT Spectrum Visualization
    fft_gt_vis = get_fft_spectrum(s_t_gt).repeat(1, 3, 1, 1)
    fft_pred_vis = get_fft_spectrum(s_t_pred).repeat(1, 3, 1, 1)
    fft_gt_vis = fft_gt_vis / (fft_gt_vis.max() + 1e-8)
    fft_pred_vis = fft_pred_vis / (fft_pred_vis.max() + 1e-8)

    # 7. Stack the 9-Row Forensic Grid
    full_vis_stack = torch.cat([
        s_h,            # Row 1: Original Hazy
        s_c,            # Row 2: Clean GT
        vis_dcp_t,      # Row 3: Input DCP Prior (What the model started with)
        vis_gt_t,       # Row 4: Target Transmission
        vis_pred_t,     # Row 5: Model's Refined Transmission
        err_edges,      # Row 6: Edge Error (Where model misses)
        fft_gt_vis,     # Row 7: Target FFT
        fft_pred_vis,   # Row 8: Model FFT
        s_h_recon       # Row 9: Physics Reconstruction Check
    ], dim=0)

    # 8. Log and Print
    grid = torchvision.utils.make_grid(full_vis_stack, nrow=4, normalize=False)
    writer.add_image(f'benchmark/{v_name}', grid, 0)
    
    metrics = {
        "Recon_PSNR": recon_psnr.item(),
        "Trans_PSNR": trans_psnr.item(),
        "Edge_PSNR": edge_psnr.item(),
        "Mean_A": s_A_spatial.mean().item()
    }
    
    for k, v in metrics.items():
        writer.add_scalar(f"Benchmark/{k}", v, 0)
    
    print(f"\n--- {v_name} BENCHMARK ---")
    print(f"Recon PSNR (Physics): {metrics['Recon_PSNR']:.2f} dB")
    print(f"Trans PSNR (Total)  : {metrics['Trans_PSNR']:.2f} dB")
    print(f"Edge PSNR (Sharp)   : {metrics['Edge_PSNR']:.2f} dB")
    
    writer.close()
    return metrics


checkpoint_path = "checkpoints/DCP_Refine_V3_VariantD_wt1.0_wr0.5_ws1.5_ep29.pth"

benchmark_dcp_residual_model(checkpoint_path, val_loader, log_base="runs/Benchmarks/DCP_Residual")


--- DCP_V2_Residual_Refinement BENCHMARK ---
Recon PSNR (Physics): 26.39 dB
Trans PSNR (Total)  : 20.95 dB
Edge PSNR (Sharp)   : 34.31 dB


{'Recon_PSNR': 26.392066955566406,
 'Trans_PSNR': 20.948898315429688,
 'Edge_PSNR': 34.30743408203125,
 'Mean_A': 0.8512442111968994}

In [50]:
def benchmark_dcp_residual_model(checkpoint_path, val_loader, log_base="runs/Benchmarks/DCP_Residual"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Setup Naming
    v_name = "DCP_Refine_V3"
    writer = SummaryWriter(log_dir=os.path.join(log_base, v_name))

    # 2. Initialize Models
    dcp_estimator = BatchedDCP(patch_size=15).to(device)
    dcp_estimator.eval()
    
    model = AblationPhysicsEstimator(in_channels=4, base_dim=32).to(device)
    
    # Load weights safely
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    # 3. Get Fixed Batch for Visualization (Seed 42 for consistency)
    g = torch.Generator()
    g.manual_seed(42) 
    fixed_loader = DataLoader(val_loader.dataset, batch_size=4, shuffle=True, generator=g)
    s_h, s_c, s_t_gt = next(iter(fixed_loader))
    s_h, s_c, s_t_gt = s_h.to(device), s_c.to(device), s_t_gt.to(device)

    # 4. Inference & Metric Calculation
    with torch.no_grad():
        # A. Compute Prior & Prediction
        s_t_dcp, _ = dcp_estimator(s_h)
        s_t_pred, s_A_spatial = model(s_h, s_t_dcp)
        s_t_pred = torch.clamp(s_t_pred, 0.0, 1.0)
        s_A_spatial = torch.clamp(s_A_spatial, 0.0, 1.0)
        
        # B. Run full evaluation suite (PSNR, SSIM, GMSD, TV)
        quality_metrics = evaluate_transmission_quality(s_t_pred, s_t_gt)
        
        # C. Physical Reconstruction Check
        A_eff = s_A_spatial.view(-1, 3, 1, 1) if s_A_spatial.dim() == 2 else s_A_spatial
        s_h_recon = torch.clamp(s_c * s_t_pred + A_eff * (1.0 - s_t_pred), 0, 1)

        # D. Reconstruction Metrics
        recon_mse = F.mse_loss(s_h_recon, s_h)
        recon_psnr = 10 * torch.log10(1.0 / (recon_mse + 1e-8))

    # 5. Build Diagnostic Rows
    vis_dcp_t = s_t_dcp.repeat(1, 3, 1, 1)
    vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
    vis_pred_t = s_t_pred.repeat(1, 3, 1, 1)

    abs_err = torch.abs(s_t_pred - s_t_gt)
    err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)
    err_edges = err_edges / (err_edges.max() + 1e-8)

    fft_gt_vis = get_fft_spectrum(s_t_gt).repeat(1, 3, 1, 1)
    fft_pred_vis = get_fft_spectrum(s_t_pred).repeat(1, 3, 1, 1)

    # 6. Stack the 9-Row Forensic Grid
    full_vis_stack = torch.cat([
        s_h,            # Row 1: Original Hazy
        s_c,            # Row 2: Clean GT
        vis_dcp_t,      # Row 3: Input DCP Prior
        vis_gt_t,       # Row 4: Target Transmission
        vis_pred_t,     # Row 5: Model's Refined Transmission
        err_edges,      # Row 6: Edge Error
        fft_gt_vis,     # Row 7: Target FFT
        fft_pred_vis,   # Row 8: Model FFT
        s_h_recon       # Row 9: Physics Reconstruction Check
    ], dim=0)

    # 7. Log and Print
    grid = torchvision.utils.make_grid(full_vis_stack, nrow=4, normalize=False)
    writer.add_image(f'benchmark/{v_name}_Forensics', grid, 0)
    
    # Consolidate all metrics
    final_metrics = {
        "Recon_PSNR": recon_psnr.item(),
        "Trans_PSNR": quality_metrics["PSNR (HIGH is good)"],
        "Trans_SSIM": quality_metrics["SSIM (HIGH is good)"],
        "Trans_GMSD": quality_metrics["GMSD (LOW is good)"],
        "Smoothness_TV": quality_metrics["TV_Smoothness"],
        "Mean_Atmosphere": s_A_spatial.mean().item()
    }
    
    for k, v in final_metrics.items():
        writer.add_scalar(f"Benchmark/{k}", v, 0)
    
    print(f"\n--- {v_name} BENCHMARK ---")
    print(f"Physics Recon PSNR : {final_metrics['Recon_PSNR']:.2f} dB")
    print(f"Transmission PSNR  : {final_metrics['Trans_PSNR']:.2f} dB")
    print(f"Transmission SSIM  : {final_metrics['Trans_SSIM']:.4f}")
    print(f"Transmission GMSD  : {final_metrics['Trans_GMSD']:.4f} (Lower is better)")
    
    writer.close()
    return final_metrics

checkpoint_path = "checkpoints/DCP_Refine_V3_VariantD_wt1.0_wr0.5_ws1.5_ep29.pth"

benchmark_dcp_residual_model(checkpoint_path, val_loader, log_base="runs/Benchmarks/DCP_Residual")


--- DCP_Refine_V3 BENCHMARK ---
Physics Recon PSNR : 26.39 dB
Transmission PSNR  : 20.95 dB
Transmission SSIM  : 0.9267
Transmission GMSD  : 0.0886 (Lower is better)


{'Recon_PSNR': 26.392066955566406,
 'Trans_PSNR': 20.948902130126953,
 'Trans_SSIM': 0.92667555809021,
 'Trans_GMSD': 0.08856723457574844,
 'Smoothness_TV': 5654.18359375,
 'Mean_Atmosphere': 0.8512442111968994}

### Temporary Saving here